# Phase 4 — The Routing Agent

This notebook loads your trained LSTM and builds the decision-support logic:
1. Load the model, scaler, and region encoding from Phase 3
2. Pull recent data to build a live input sequence per region
3. Given a flight route, generate candidate deviations
4. Predict risk for each candidate, estimate cost (distance/fuel/time)
5. Rank and recommend, with a plain-English explanation

**Known limitation (by design, for this prototype):** our regions are 6 large boxes covering the whole US. A deviation that stays inside the same box won't change the model's risk prediction — a production version would use a much finer grid (e.g. 0.5° cells).

## Step 1 — Setup: load model, scaler, and Supabase credentials

In [ ]:
import requests
import pandas as pd
import numpy as np
import pickle
import math
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/flight_corridor_project'

import tensorflow as tf
model = tf.keras.models.load_model(f'{SAVE_DIR}/turbulence_lstm.keras')

with open(f'{SAVE_DIR}/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open(f'{SAVE_DIR}/region_columns.pkl', 'rb') as f:
    region_columns = pickle.load(f)

SEQ_LEN = 6  # must match what Phase 3 used

SUPABASE_URL = "........"   # <-- same as before
SUPABASE_KEY = "..........."                    # <-- same as before

headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
}

print("Model, scaler, and region encoding loaded.")
print("Regions the model knows about:", region_columns)

Mounted at /content/drive
Model, scaler, and region encoding loaded.
Regions the model knows about: ['region_MW', 'region_NE', 'region_NW', 'region_SE', 'region_SW', 'region_SWC']


## Step 2 — Region geometry (same boxes as before, plus centroids for distance math)

In [ ]:
region_boxes = {
    "NE":  (35, -90, 50, -65),
    "SE":  (24, -90, 35, -65),
    "MW":  (35, -105, 50, -90),
    "SW":  (24, -105, 35, -90),
    "NW":  (35, -125, 50, -105),
    "SWC": (24, -125, 35, -105),
}

def get_region(lat, lon):
    for name, (minLat, minLon, maxLat, maxLon) in region_boxes.items():
        if minLat <= lat <= maxLat and minLon <= lon <= maxLon:
            return name
    return None

def haversine_nm(lat1, lon1, lat2, lon2):
    # Great-circle distance in nautical miles
    R_nm = 3440.065
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlambda/2)**2
    return 2 * R_nm * math.asin(math.sqrt(a))

## Step 3 — Pull recent data and build a live input sequence per region

This mirrors Phase 3's hourly-grid logic, but only needs the last `SEQ_LEN` hours — this is the "what's happening right now" snapshot the model needs to forecast forward.

In [ ]:
def fetch_recent_pireps(hours_back=24, page_size=1000):
    cutoff_ts = int((pd.Timestamp.utcnow() - pd.Timedelta(hours=hours_back)).timestamp())
    all_rows = []
    offset = 0
    while True:
        params = {
            "select": "*",
            "order": "obs_time.asc",
            "limit": page_size,
            "offset": offset,
            "obs_time": f"gte.{cutoff_ts}"
        }
        resp = requests.get(f"{SUPABASE_URL}/rest/v1/pireps", headers=headers, params=params)
        if resp.status_code != 200:
            print("Fetch failed:", resp.status_code, resp.text[:300])
            break
        batch = resp.json()
        if not batch:
            break
        all_rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    return pd.DataFrame(all_rows)

recent_df = fetch_recent_pireps(hours_back=24)
print("Recent rows pulled:", len(recent_df))

if len(recent_df) > 0:
    recent_df["obs_datetime"] = pd.to_datetime(recent_df["obs_time"], unit="s")
    recent_df["region"] = recent_df.apply(lambda r: get_region(r["lat"], r["lon"]), axis=1)
    recent_df = recent_df.dropna(subset=["region"])
    recent_df["hour"] = recent_df["obs_datetime"].dt.floor("h")

    recent_hourly = recent_df.groupby(["region", "hour"])["turb_score"].max().reset_index()
else:
    recent_hourly = pd.DataFrame(columns=["region", "hour", "turb_score"])
    print("No recent data — predictions below will fall back to zero-risk input sequences.")

Recent rows pulled: 1895


In [ ]:
def get_recent_sequence(region, seq_len=SEQ_LEN):
    """Return the last seq_len hourly (scaled) turbulence scores for a region,
    padding with 0 (no data = assume calm) if we don't have enough recent history yet."""
    sub = recent_hourly[recent_hourly["region"] == region].sort_values("hour")
    scores = sub["turb_score"].values[-seq_len:] if len(sub) > 0 else np.array([])

    if len(scores) < seq_len:
        pad = np.zeros(seq_len - len(scores))
        scores = np.concatenate([pad, scores])

    scaled = scaler.transform(scores.reshape(-1, 1)).flatten()
    return scaled

def predict_region_risk(region):
    """Predict next-hour turbulence risk (0-8 scale) for a region using the trained LSTM."""
    seq = get_recent_sequence(region)
    region_onehot = [1.0 if col == f"region_{region}" else 0.0 for col in region_columns]
    seq_features = np.array([[s] + region_onehot for s in seq])
    seq_features = seq_features.reshape(1, SEQ_LEN, -1)

    pred_scaled = model.predict(seq_features, verbose=0)[0][0]
    pred_scaled = np.clip(pred_scaled, 0, 1)
    pred_score = scaler.inverse_transform([[pred_scaled]])[0][0]
    return max(0.0, pred_score)

# Quick test: predicted risk for every region right now
print("Current predicted next-hour turbulence risk by region:")
for r in region_boxes.keys():
    print(f"  {r}: {predict_region_risk(r):.2f} / 8")

Current predicted next-hour turbulence risk by region:


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


  NE: 2.51 / 8
  SE: 1.84 / 8
  MW: 2.59 / 8


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


  SW: 1.23 / 8
  NW: 1.66 / 8
  SWC: 1.69 / 8


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


## Step 4 — Define a flight route and generate candidate deviations

Given an origin and destination, we build a straight-line path of waypoints, then generate alternates by shifting the path north/south by a few degrees of latitude — a simplified stand-in for real deviation vectors an aircraft could actually fly.

In [ ]:
def generate_waypoints(origin, destination, n_points=5):
    """origin/destination: (lat, lon) tuples. Returns n_points evenly spaced waypoints."""
    lat1, lon1 = origin
    lat2, lon2 = destination
    return [
        (lat1 + (lat2 - lat1) * t, lon1 + (lon2 - lon1) * t)
        for t in np.linspace(0, 1, n_points)
    ]

def shift_route(waypoints, lat_offset):
    """Shift every waypoint north (positive) or south (negative) by lat_offset degrees,
    except keep the first and last waypoint fixed (you still depart/arrive at the same airports)."""
    shifted = [waypoints[0]]
    for lat, lon in waypoints[1:-1]:
        shifted.append((lat + lat_offset, lon))
    shifted.append(waypoints[-1])
    return shifted

def route_extra_distance_nm(original, shifted):
    """Sum of leg distances for the shifted route minus the original route."""
    def path_length(wpts):
        return sum(haversine_nm(wpts[i][0], wpts[i][1], wpts[i+1][0], wpts[i+1][1]) for i in range(len(wpts)-1))
    return path_length(shifted) - path_length(original)

## Step 5 — Evaluate and rank candidate routes

For each candidate: predict the **worst** (max) turbulence risk across its waypoints' regions, and estimate the fuel/time cost of any added distance.

In [ ]:
FUEL_KG_PER_NM = 6.5   # rough A320 cruise fuel burn estimate, kg per nautical mile
SPEED_KTS = 470        # rough A320 cruise ground speed, knots (nm per hour)

def evaluate_route(label, waypoints, original_waypoints):
    regions_hit = list(set(get_region(lat, lon) for lat, lon in waypoints if get_region(lat, lon)))
    if not regions_hit:
        risk = 0.0
    else:
        risk = max(predict_region_risk(r) for r in regions_hit)

    extra_nm = route_extra_distance_nm(original_waypoints, waypoints)
    extra_fuel_kg = max(0, extra_nm) * FUEL_KG_PER_NM
    extra_time_min = max(0, extra_nm) / SPEED_KTS * 60

    return {
        "label": label,
        "regions": regions_hit,
        "predicted_risk": round(risk, 2),
        "extra_distance_nm": round(extra_nm, 1),
        "extra_fuel_kg": round(extra_fuel_kg, 1),
        "extra_time_min": round(extra_time_min, 1),
    }

def rank_candidates(origin, destination, deviations_deg=(0, 2, -2, 4, -4)):
    original_wpts = generate_waypoints(origin, destination)
    results = []
    for dev in deviations_deg:
        label = "Direct route" if dev == 0 else f"{'North' if dev > 0 else 'South'} deviation ({abs(dev)}° lat)"
        candidate_wpts = shift_route(original_wpts, dev) if dev != 0 else original_wpts
        results.append(evaluate_route(label, candidate_wpts, original_wpts))

    # Simple composite score: risk matters most, cost is a tiebreaker.
    # Weighting is a design choice — tune RISK_WEIGHT / COST_WEIGHT for your priorities.
    RISK_WEIGHT = 10
    COST_WEIGHT = 0.01
    for r in results:
        r["composite_score"] = RISK_WEIGHT * r["predicted_risk"] + COST_WEIGHT * r["extra_fuel_kg"]

    results.sort(key=lambda r: r["composite_score"])
    return results

## Step 6 — Try it on an example route

Default example: JFK (New York) → LAX (Los Angeles). Change the coordinates to try other routes.

In [ ]:
JFK = (40.6413, -73.7781)
LAX = (33.9416, -118.4085)

candidates = rank_candidates(JFK, LAX)

for c in candidates:
    print(f"{c['label']:30s} | risk: {c['predicted_risk']:.2f}/8 | "
          f"+{c['extra_distance_nm']:.0f} nm | +{c['extra_fuel_kg']:.0f} kg fuel | "
          f"+{c['extra_time_min']:.1f} min | regions: {c['regions']}")

best = candidates[0]
print(f"\n Recommended: {best['label']}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/valida

Direct route                   | risk: 2.59/8 | +0 nm | +0 kg fuel | +0.0 min | regions: ['MW', 'NE', 'NW', 'SWC']
North deviation (2° lat)       | risk: 2.59/8 | +-18 nm | +0 kg fuel | +0.0 min | regions: ['MW', 'NE', 'NW', 'SWC']
North deviation (4° lat)       | risk: 2.59/8 | +12 nm | +76 kg fuel | +1.5 min | regions: ['MW', 'NE', 'NW', 'SWC']
South deviation (2° lat)       | risk: 2.59/8 | +67 nm | +438 kg fuel | +8.6 min | regions: ['MW', 'NE', 'SWC']
South deviation (4° lat)       | risk: 2.51/8 | +179 nm | +1162 kg fuel | +22.8 min | regions: ['SWC', 'NE', 'SE', 'SW']

 Recommended: Direct route


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


## Step 7 — Generate a plain-English recommendation

This is the "agentic reasoning" layer from your original idea — turning the ranked numbers into a decision-support message, the way a real dispatch system would present it to a pilot.

In [ ]:
def explain_recommendation(candidates):
    best = candidates[0]
    direct = next((c for c in candidates if c["label"] == "Direct route"), None)

    lines = []
    lines.append(f"RECOMMENDATION: {best['label']}")
    lines.append(f"Predicted turbulence risk: {best['predicted_risk']:.1f} / 8")

    if direct and best["label"] != "Direct route":
        risk_reduction = direct["predicted_risk"] - best["predicted_risk"]
        if risk_reduction > 0:
            lines.append(
                f"This reduces predicted turbulence risk by {risk_reduction:.1f} points vs. the direct route, "
                f"at a cost of {best['extra_distance_nm']:.0f} nm (+{best['extra_fuel_kg']:.0f} kg fuel, "
                f"+{best['extra_time_min']:.1f} min)."
            )
        else:
            lines.append("The direct route already has the lowest predicted risk among evaluated candidates.")
    elif best["label"] == "Direct route":
        lines.append("The direct route has the lowest predicted risk — no deviation recommended.")

    lines.append("\nAll evaluated options:")
    for c in candidates:
        lines.append(
            f"  • {c['label']}: risk {c['predicted_risk']:.1f}/8, "
            f"+{c['extra_fuel_kg']:.0f} kg, +{c['extra_time_min']:.1f} min"
        )

    lines.append("\nNote: this is a decision-support recommendation only. "
                  "Any route change must be requested from and approved by ATC.")

    return "\n".join(lines)

print(explain_recommendation(candidates))

RECOMMENDATION: Direct route
Predicted turbulence risk: 2.6 / 8
The direct route has the lowest predicted risk — no deviation recommended.

All evaluated options:
  • Direct route: risk 2.6/8, +0 kg, +0.0 min
  • North deviation (2° lat): risk 2.6/8, +0 kg, +0.0 min
  • North deviation (4° lat): risk 2.6/8, +76 kg, +1.5 min
  • South deviation (2° lat): risk 2.6/8, +438 kg, +8.6 min
  • South deviation (4° lat): risk 2.5/8, +1162 kg, +22.8 min

Note: this is a decision-support recommendation only. Any route change must be requested from and approved by ATC.


## Summary

This notebook now contains the full agent pipeline: load model → predict live regional risk → generate candidate deviations → rank by risk/cost → explain the recommendation.

**Try it yourself:** change the `JFK`/`LAX` coordinates in Step 6 to other airport pairs, or adjust `deviations_deg` in Step 5 to test different deviation magnitudes.

